# 🛒 온라인 채널 제품 판매량 예측 (PyTorch LSTM)
> **모델 전략:** 전체 제품을 하나의 글로벌 LSTM으로 학습 (Global Time-Series Model)  
> **입력:** 과거 28일 시퀀스 → **출력:** 미래 21일 판매량 (Direct Multi-Step)

## 📌 프로젝트 목적
과거의 제품 판매량, 브랜드 검색량(Keyword) 등의 시계열 데이터를 활용하여 향후 21일간의 제품별 판매량을 예측하는 딥러닝 모델 구축
> **예측 대상:** 약 15,890개 제품의 **미래 21일** 일별 판매 수량  
> **학습 데이터:** 2022-01-01 ~ 2023-04-04 (459일치)  
> **예측 기간:** 2023-04-05 ~ 2023-04-25 (21일)


## 📁 데이터셋 (Data)
* `train.csv` / `sales.csv`: 일자별 제품 판매량 데이터
* `product_info.csv`: 제품 메타 정보 (대분류, 중분류, 브랜드 등)
* `brand_keyword_cnt.csv`: 일자별 브랜드 검색량 데이터
* `sample_submission.csv`: 제출 양식

> 판매수량 (train.csv), 매출액(sales.csv)으로 판매단가(price) 추출하기


---

## 🗺️ 전체 흐름

**STEP 0**   데이터 로드 & 환경 설정 <br>

**STEP 1**   EDA (탐색적 데이터 분석)<br>
1-1. 기본 통계<br>
1-2. 일별 트렌드<br>
1-3. 요일/월 패턴<br>
1-4. 대분류별 판매<br>
1-5. 샘플 제품 시계열<br>
1-6. ★ 0의 두 가지 종류 구분 (구조적 0 vs 진짜 0)<br>
1-7. ★ 안 팔리는 제품 비율 분석<br>

**STEP 2**   전처리 & 피처 엔지니어링<br>
2-1. Wide -> Long 변환<br>
2-2. 카테고리 인코딩<br>
2-3. 키워드 병합 + 유효성 검증 (시차 상관관계)<br>
2-4. 날짜 피처<br>
2-5. Log 변환 + ★ product_info 텍스트 키워드 피처<br>
2-6. Lag & Rolling 피처<br>
2-7. 최종 피처 목록<br>
2-8. 스케일링<br>

**STEP 3**   LSTM 모델 구성<br>

**STEP 4**   컴파일 (MAE Loss, Adam)<br>

**STEP 5**   Callbacks (EarlyStopping, ReduceLR)<br>

**STEP 6**   학습<br>

**STEP 7**   시각화<br>

**STEP 8**   최종 예측 & 제출<br>

---

## 🤔 왜 LSTM?

| 방법 | 장점 | 단점 |
|---|---|---|
| 단순 이동평균 | 빠르고 직관적 | 복잡한 패턴 포착 불가 |
| ARIMA (통계) | 해석 쉬움 | 15,890개에 각각 적용 -> 너무 느림 |
| XGBoost | 빠르고 강력 | 시간 순서 정보를 직접 활용 못함 |
| **LSTM ** | **시간 순서 패턴 학습에 최적화** | 학습 시간이 좀 걸림 |
> LSTM은 '오래된 기억도 잊지 않는' 특수한 RNN 구조
> '지난주 월요일에 많이 팔렸으니, 이번 주 월요일도 많이 팔릴 것' 같은 패턴을 스스로 학습

---

## 🏗️ 모델 전략: 글로벌 모델

- **개별 모델** (제품마다 모델 1개): 15,890개 x 학습시간 = 며칠이 걸림 ==> ❌
- **글로벌 모델** (모든 제품을 하나의 모델로): 빠르고, 제품 간 공통 패턴도 학습 ==> ✅
  -> 데이터가 적은 신제품도 다른 제품의 패턴에서 배울 수 있음






## 

| 결정 | 선택 | 이유 |
|---|---|---|
| 모델 구조 | 글로벌 LSTM (전체 제품 하나의 모델) | 15,890개 개별 모델은 시간상 불가 |
| 0 처리 | 구조적 0 제외, 진짜 0 유지 | 출시 전 기간의 0은 패턴이 아님 |
| 텍스트 피처 | 키워드 이진 추출 | BERT 임베딩보다 빠르고 해석 가능 |
| 키워드 신호 | 시차 상관관계로 유효성 검증 후 결정 | 브랜드 단위라 노이즈 가능성 있음 |
| 예측 방식 | Direct Multi-Step (28일 -> 21일) | 재귀 예측보다 오차 누적 없음 |


## STEP 0 — 라이브러리 불러오기 & 데이터 로드

### 사용하는 주요 라이브러리
| 라이브러리 | 역할 |
|---|---|
| `pandas` | 표(DataFrame) 형태로 데이터 다루기 |
| `numpy` | 수치 계산, 배열 연산 |
| `matplotlib / seaborn` | 그래프 그리기 |
| `sklearn` | 데이터 전처리 (스케일링, 인코딩) |
| `torch (PyTorch)` | 딥러닝 모델 만들기 & 학습 |
| `tqdm` | 학습 진행 바 표시 |


In [ ]:
# ============================================================
# 📦 필요한 라이브러리 전부 불러오기
# ============================================================

import os           # 파일 경로, 폴더 생성 등 운영체제 관련 기능
import warnings     # 불필요한 경고 메시지 숨기기
import platform

# 수치 계산
import numpy as np  # 배열 연산의 기본 (np.mean, np.array 등)
import pandas as pd # 표(DataFrame) 형태 데이터 다루기

# 시각화
import matplotlib.pyplot as plt     # 기본 그래프
import matplotlib
import seaborn as sns               # 더 예쁜 통계 그래프
import matplotlib.font_manager as fm

# 전처리 도구
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
# LabelEncoder: 텍스트 카테고리 -> 숫자 (예: '대분류A' -> 0, '대분류B' -> 1)
# MinMaxScaler: 숫자를 0~1 범위로 압축 (모델이 학습하기 좋은 형태)
from sklearn.metrics import mean_absolute_error  # MAE 계산 (예측 오차 측정)

# 딥러닝 (PyTorch)
import torch                            # PyTorch 핵심
import torch.nn as nn                   # 신경망 레이어들 (LSTM, Linear 등)
from torch.utils.data import Dataset, DataLoader
# Dataset   : 우리 데이터를 PyTorch가 읽을 수 있는 형태로 포장
# DataLoader: Dataset을 배치(batch) 단위로 잘라서 학습에 공급

from tqdm.notebook import tqdm  # Jupyter에서 예쁘게 보이는 진행 바

# ============================================================
# ⚙️ 기본 설정
# ============================================================

warnings.filterwarnings('ignore')  # 경고 메시지 숨기기

# 한글 깨짐 방지 (Windows 환경)
matplotlib.rcParams['axes.unicode_minus'] = False  # 마이너스 부호 깨짐 방지

# ============================================================
# 📁 경로 설정
# ============================================================

HOME = os.getcwd()  # 현재 작업 폴더 = '/mnt/c/users/min2m/github/Sales/권희민'
DATA_DIR   = os.path.join(HOME, 'data')    # 데이터 파일들이 있는 폴더
OUTPUT_DIR = os.path.join(HOME, 'output')  # 결과물을 저장할 폴더

os.makedirs(OUTPUT_DIR, exist_ok=True)  # output 폴더가 없으면 자동으로 만들기

print(f'📁 홈 경로    : {HOME}')
print(f'📁 데이터 경로: {DATA_DIR}')
print(f'📁 출력 경로  : {OUTPUT_DIR}')

# ============================================================
# 🖥️ GPU 설정
# ============================================================

# GPU가 있으면 GPU를, 없으면 CPU를 자동으로 선택
# GPU: 행렬 연산을 병렬 처리해서 딥러닝 학습을 수십 배 빠르게 함
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\n🖥️  사용 디바이스: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'   GPU 이름     : {torch.cuda.get_device_name(0)}')
    print(f'   GPU 메모리   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


In [ ]:
# 폰트 설정 

def set_korean_font():
    """Windows / Linux(Colab 포함) / macOS 환경에 맞춰 한글 폰트를 설정합니다."""
    
    system = platform.system()

    if system == "Windows":
        # Windows는 기본 맑은 고딕 사용
        font_path = "c:/Windows/Fonts/malgun.ttf"
        if os.path.isfile(font_path):
            fm.fontManager.addfont(font_path)
            font_name = fm.FontProperties(fname=font_path).get_name()
            plt.rc("font", family=font_name)
            print(f"한글 폰트 설정: {font_name} ({font_path})")
        else:
            print("Windows용 맑은 고딕 폰트를 찾을 수 없습니다.")

    elif system == "Linux":
        # 1) 이미 설치된 나눔 폰트 탐색
        nanum_paths = [
            "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
            "/usr/share/fonts/nanum/NanumGothic.ttf",
            os.path.expanduser("~/.local/share/fonts/NanumGothic.ttf"),
        ]
        font_path = next((p for p in nanum_paths if os.path.isfile(p)), None)

        # 2) 없으면 apt-get으로 설치 시도 (Colab / Ubuntu)
        if font_path is None:
            try:
                print("나눔 폰트 설치 중 (apt-get)...")
                import subprocess
                subprocess.run(
                    ["apt-get", "install", "-y", "-qq", "fonts-nanum"],
                    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                )
                # 설치 후 다시 탐색
                font_path = next((p for p in nanum_paths if os.path.isfile(p)), None)
            except Exception:
                pass

        # 3) apt-get 불가 시 직접 다운로드 (로컬 환경 등)
        if font_path is None:
            print("나눔 폰트 다운로드 중...")
            import urllib.request
            font_dir = os.path.expanduser("~/.local/share/fonts")
            os.makedirs(font_dir, exist_ok=True)
            font_path = os.path.join(font_dir, "NanumGothic.ttf")
            url = "https://github.com/google/fonts/raw/main/ofl/nanumgothic/NanumGothic-Regular.ttf"
            urllib.request.urlretrieve(url, font_path)

        if font_path and os.path.isfile(font_path):
            fm.fontManager.addfont(font_path)
            font_name = fm.FontProperties(fname=font_path).get_name()
            plt.rc("font", family=font_name)
            print(f"한글 폰트 설정: {font_name} ({font_path})")
        else:
            print("경고: 한글 폰트를 설정할 수 없습니다.")

    elif system == "Darwin":  # macOS
        plt.rc("font", family="AppleGothic")
        print("한글 폰트 설정: AppleGothic")

    # 마이너스 기호 깨짐 방지
    plt.rc("axes", unicode_minus=False)

# 함수 실행
set_korean_font()

In [ ]:
# ============================================================
# 📂 데이터 로드
# ============================================================

# pd.read_csv(): CSV 파일을 읽어서 DataFrame(표)으로 변환
train      = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
sales      = pd.read_csv(os.path.join(DATA_DIR, 'sales.csv'))
product    = pd.read_csv(os.path.join(DATA_DIR, 'product_info.csv'))
keyword    = pd.read_csv(os.path.join(DATA_DIR, 'brand_keyword_cnt.csv'))
submission = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))

# ============================================================
# 📜 컬럼 분류
# ============================================================
# train.csv는 두 종류의 컬럼으로 나뉨:
#   META_COLS : 제품 정보 (ID, 카테고리, 브랜드) - 날짜와 무관한 고정 정보
#   DATE_COLS : 날짜별 판매량 컬럼들 (2022-01-01, 2022-01-02, ...)
META_COLS = ['ID', '제품', '대분류', '중분류', '소분류', '브랜드']
DATE_COLS = [c for c in train.columns if c not in META_COLS]  # 나머지 = 날짜 컬럼

# 제출 파일에 있는 날짜 = 예측할 미래 21일
PRED_DATES = [c for c in submission.columns if c != 'ID']

print('=' * 55)
print(f'📊 train.csv   : {train.shape[0]:,}행 x {train.shape[1]}열')
print(f'📊 sales.csv   : {sales.shape[0]:,}행 x {sales.shape[1]}열')
print(f'📊 product.csv : {product.shape[0]:,}행 x {product.shape[1]}열')
print(f'📊 keyword.csv : {keyword.shape[0]:,}행 x {keyword.shape[1]}열')
print('=' * 55)
print(f'📅 학습 기간   : {DATE_COLS[0]}  ~  {DATE_COLS[-1]}  ({len(DATE_COLS)}일)')
print(f'🎯 예측 기간   : {PRED_DATES[0]}  ~  {PRED_DATES[-1]}  ({len(PRED_DATES)}일)')
print('=' * 55)
print()
print('[train.csv 3행 미리보기]')
train[META_COLS + DATE_COLS[:3]].head(3)

In [ ]:
train.info()

In [ ]:
train.describe()

In [ ]:
def check_data_summary(df):
    summary = pd.DataFrame({
        'Data Type': df.dtypes,
        'Null Count': df.isnull().sum(),
        'Null (%)': (df.isnull().sum() / len(df)) * 100,
        'Unique Values': df.nunique()
    })
    return summary

print(check_data_summary(train))

> ID/제품 : 일부 중복 제품이 있나?

> 대중소분류 : 범주형 변수로 모델에 학습시키기

> 날짜 컬럼 : 결측치 0. 모든 판매량 숫자가 일단 채워져 있음

## STEP 1 — EDA (탐색적 데이터 분석)

### EDA (Exploratory Data Analysis)

1. **데이터 크기와 구조** — 얼마나 많은 제품이 있나? 기간은 어떻게 되나?
2. **결측치** — 비어있는 데이터가 있나? 있다면 채워야 함
3. **분포** — 판매량이 0인 날이 많은가? 극단적인 최대값이 있는가?
4. **시간 패턴** — 계절성? 요일 패턴? -> STEP 2 피처 엔지니어링에 활용
5. **이상치** — 갑자기 폭발적으로 팔린 날이 있는가?

In [ ]:
# ============================================================
# 1-1. 기본 통계 정보
# ============================================================

print('=== 📌 제품 카테고리 구조 ===')
print(f'  전체 제품 수   : {train["제품"].nunique():,}개')
print(f'  대분류 수      : {train["대분류"].nunique()}개')
print(f'  중분류 수      : {train["중분류"].nunique()}개')
print(f'  소분류 수      : {train["소분류"].nunique()}개')
print(f'  브랜드 수      : {train["브랜드"].nunique():,}개')
print()

# .values: DataFrame -> numpy 배열로 변환
# .flatten(): 2차원 배열(행렬)을 1차원 배열로 쭉 펼치기
qty_values = train[DATE_COLS].values.flatten()

print('=== 📌 판매 수량 분포 ===')
print(f'  결측치(NaN) 수   : {np.isnan(qty_values).sum()}개')
print(f'  0인 날 비율      : {(qty_values == 0).mean():.1%}')  # 판매 없는 날 비율
print(f'  최댓값           : {qty_values.max():,.0f}개')
print(f'  평균 (판매>0 만) : {qty_values[qty_values > 0].mean():.2f}개')
print(f'  중앙값 (전체)    : {np.median(qty_values):.1f}개')

# 💡 0이 많다는 건? 대부분의 제품이 매일 팔리지 않음 (희소한 데이터)
# -> 이걸 감안해서 모델을 설계해야 함


> 전체 판매 데이터 중 0인 날이 63.8% (절반이상!!!)

> 0이 진짜 안팔린건지, 출시전 제품인지, 품절상태여서 못판건지...? 품절이라면 결측치임 (DL에서는 이런것도 알아서 처리해주나? 기간을 봐야겠지만)

In [ ]:
# ============================================================
# 1-2. 전체 일별 판매량 트렌드 시각화
# ============================================================
# 왜? -> 전체적으로 판매가 증가하는지, 계절적 패턴이 있는지 확인

# 모든 제품의 일별 판매량을 합산
# axis=0: 행(제품) 방향으로 합산 -> 날짜별 전체 판매량 합계
daily_total = train[DATE_COLS].sum(axis=0)
daily_total.index = pd.to_datetime(daily_total.index)  # 날짜 문자열 -> 날짜 타입

fig, axes = plt.subplots(2, 1, figsize=(16, 8))

# 위 그래프: 원본 일별 합계
axes[0].plot(daily_total.index, daily_total.values, linewidth=0.8, color='steelblue')
axes[0].set_title('전체 제품 일별 판매량 합계 (원본)', fontsize=14)
axes[0].set_ylabel('판매량')
axes[0].grid(alpha=0.3)

# 아래 그래프: 7일 이동평균과 비교
# 이동평균(rolling mean): 최근 N일 데이터의 평균
# 일별 노이즈를 제거하고 전체적인 추세(trend)를 보여줌
rolling7 = daily_total.rolling(window=7).mean()
axes[1].plot(daily_total.index, daily_total.values,
             linewidth=0.5, alpha=0.4, color='steelblue', label='일별 실제값')
axes[1].plot(rolling7.index, rolling7.values,
             linewidth=2, color='tomato', label='7일 이동평균 (추세)')
axes[1].set_title('7일 이동평균으로 본 판매 트렌드', fontsize=14)
axes[1].set_ylabel('판매량')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


> 특정 시기에 급등하는 구간? -> 이벤트/프로모션 영향??

> 전반적으로 증가/감소 추세인가? -> 트렌드 피처가 필요할 수도

In [ ]:
# ============================================================
# 1-3. 요일별 & 월별 판매 패턴
# ============================================================
# 왜? -> 요일/월 패턴이 있다면 STEP 2에서 요일/월 피처를 만들어야 함!

daily_df = daily_total.reset_index()
daily_df.columns = ['date', 'qty']
daily_df['요일'] = daily_df['date'].dt.dayofweek  # 0=월요일, 6=일요일
dow_map = {0:'월', 1:'화', 2:'수', 3:'목', 4:'금', 5:'토', 6:'일'}
daily_df['요일명'] = daily_df['요일'].map(dow_map)
daily_df['월'] = daily_df['date'].dt.month

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# boxplot: 중앙값, 사분위범위, 이상치를 한눈에 보여주는 통계 그래프
order = ['월', '화', '수', '목', '금', '토', '일']
sns.boxplot(data=daily_df, x='요일명', y='qty', order=order,
            ax=axes[0], palette='Blues')
axes[0].set_title('요일별 판매량 분포', fontsize=13)
axes[0].set_xlabel('요일')
axes[0].set_ylabel('일별 총 판매량')

sns.boxplot(data=daily_df, x='월', y='qty', ax=axes[1], palette='Greens')
axes[1].set_title('월별 판매량 분포', fontsize=13)
axes[1].set_xlabel('월')
axes[1].set_ylabel('일별 총 판매량')

plt.tight_layout()
plt.show()

# 💡 관찰 포인트:
#   - 특정 요일에 일관되게 높은가? (예: 주말 > 평일)
#   - 특정 달에 높은가? (예: 11월 블랙프라이데이, 12월 연말)
#   -> '있다'면 STEP 2에서 요일/월/공휴일 피처가 중요해짐


In [ ]:
# ============================================================
# 1-4. 대분류별 판매 비중
# ============================================================
# 왜? -> 어떤 카테고리가 판매의 대부분을 차지하는지 파악
#       카테고리 피처가 예측에 중요한 신호가 됨을 확인

# groupby: 대분류별로 묶기
# .sum().sum(axis=1): 날짜 합산 -> 제품 합산 -> 대분류별 총합
cat_total = (
    train.groupby('대분류')[DATE_COLS]
    .sum()
    .sum(axis=1)
    .sort_values(ascending=False)
)

plt.figure(figsize=(12, 5))
cat_total.head(20).plot(kind='bar', color='steelblue', alpha=0.8, edgecolor='white')
plt.title('대분류별 총 판매량 (상위 20개)', fontsize=14)
plt.ylabel('총 판매량')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 1-5. 샘플 제품 시계열 시각화
# ============================================================
# 왜? -> 실제로 제품 하나하나의 판매 패턴이 어떻게 생겼는지 확인
#       패턴을 보고 입력 시퀀스 길이(SEQ_LEN=28) 결정에 도움

# 총 판매량 기준 상위 6개 제품 선택
top6_idx = train[DATE_COLS].sum(axis=1).nlargest(6).index
sample_rows = train.iloc[top6_idx]

fig, axes = plt.subplots(3, 2, figsize=(16, 10))

for ax, (_, row) in zip(axes.flatten(), sample_rows.iterrows()):
    ts = row[DATE_COLS].astype(float)
    ts.index = pd.to_datetime(ts.index)  # 날짜 타입으로 변환
    ax.plot(ts.index, ts.values, linewidth=0.9, color='steelblue')
    ax.set_title(f"{row['제품']}  ({row['브랜드']})", fontsize=9)
    ax.set_ylabel('판매량')
    ax.grid(alpha=0.3)

plt.suptitle('판매량 상위 6개 제품 시계열', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

1. 2023-03 이후 급격한 판매 하락 -> 반등
> 데이터 수집 오류? 대규모 품절? 서비스 중단?

2. 계절성 뚜렷
요일별로 판매량 차이가 보임. 주말(금~일) 폭증, 평일 줄어듦
> 7일 이동평균선이 매끄러운 걸로 보아 주 단위 패턴이 매우 강함 -> 입력 피처에 요일(day of week)정보 반드시 넣기

3. 스파이크
2022-07, 10월 판매량이 평소보다 2배 이상 튀는 구간 
> 대형 프로모션? 특정이벤트? 확인 후 피처 입력 필요해 보임

> 주기적인 패턴(계절성)이 있나? 1년 주기? 1주 주기?
> 데이터가 없다가 갑자기 시작하는 제품? -> 신제품


### 🔍 0의 두 가지 종류 — 왜 구분해야 하는가?

전체 데이터에서 0이 약 63%<br>
두 가지로 구분 필요

| 종류 | 의미 | 처리 방법 |
|---|---|---|
| **구조적 0 (Structural Zero)** | 제품 출시 전 / 단종 후 기간 | **학습에서 제외** 해야 함 |
| **진짜 0 (True Zero)** | 출시 이후인데 그 날 판매 없음 | **그대로 유지** (패턴의 일부) |


**처리 전략:**
1. 제품별 `first_sale_date` (첫 판매일) 를 찾기
2. 첫 판매일 이전 데이터는 학습에서 제외 (또는 마스킹)
3. 첫 판매일 이후의 0은 그대로 유지 (진짜 패턴)

In [ ]:
# ============================================================
# 1-6. 0의 두 가지 종류 구분 분석
# ============================================================
# 목표:
#   (A) 각 제품의 첫 판매일(first_sale_date)을 찾기
#   (B) 구조적 0 vs 진짜 0 비율 계산
#   (C) '아예 안 팔리는 제품' 비율 파악

print('=== 제품별 첫 판매일 분석 ===')

# 각 제품의 날짜별 판매량 데이터 (Wide 포맷 그대로 사용)
qty_matrix = train[DATE_COLS].values  # shape: (제품수, 날짜수)
date_arr   = pd.to_datetime(DATE_COLS)  # 날짜 배열

# ── (A) 제품별 첫 판매일 / 마지막 판매일 계산 ─────────────
# argmax: True(=1)가 처음 나오는 위치의 인덱스를 반환
# qty_matrix > 0 : 판매량이 있는 날은 True, 없으면 False

first_sale_idx = []
last_sale_idx  = []
never_sold     = []  # 한 번도 팔리지 않은 제품

for row in qty_matrix:
    has_sale = (row > 0)  # 판매 있는 날 True
    if has_sale.sum() == 0:
        # 한 번도 팔린 적 없는 제품
        never_sold.append(True)
        first_sale_idx.append(-1)
        last_sale_idx.append(-1)
    else:
        never_sold.append(False)
        first_sale_idx.append(has_sale.argmax())           # 첫 True의 위치
        last_sale_idx.append(len(row) - 1 - has_sale[::-1].argmax())  # 마지막 True의 위치

train['first_sale_date']  = [date_arr[i] if i >= 0 else pd.NaT for i in first_sale_idx]
train['last_sale_date']   = [date_arr[i] if i >= 0 else pd.NaT for i in last_sale_idx]
train['first_sale_idx']   = first_sale_idx
train['never_sold']       = never_sold

# ── (B) 전체 0 중에서 구조적 0 vs 진짜 0 비율 ─────────────
total_cells       = len(DATE_COLS) * len(train)   # 전체 (제품 x 날짜) 셀 수
structural_zeros  = 0
true_zeros        = 0
actual_sales      = 0

for idx, row in train.iterrows():
    fi = row['first_sale_idx']
    li = row['last_sale_date']
    vals = qty_matrix[idx]

    if fi == -1:  # 한 번도 안 팔린 제품
        structural_zeros += len(DATE_COLS)
        continue

    last_i = row['last_sale_date']
    # 출시 전: 구조적 0
    structural_zeros += fi
    # 출시 이후 ~ 마지막 판매일: 진짜 0 + 실제 판매
    active_period = vals[fi:]
    true_zeros    += (active_period == 0).sum()
    actual_sales  += (active_period > 0).sum()

print(f'  전체 셀 수          : {total_cells:,}')
print(f'  구조적 0 (출시 전)  : {structural_zeros:,}  ({structural_zeros/total_cells:.1%})')
print(f'  진짜 0 (출시 후 미판매): {true_zeros:,}  ({true_zeros/total_cells:.1%})')
print(f'  실제 판매 데이터    : {actual_sales:,}  ({actual_sales/total_cells:.1%})')


In [ ]:
# ============================================================
# 1-7. '아예 안 팔리는 제품' 비율 
# ============================================================
# 한 번도 팔린 적 없는 제품을 모델링해야 할까?
# -> 예측해봤자 0밖에 나올 수 없으니, 모델에서 제외하는 게 효율적
# -> 제출 파일에는 0으로 채우면 됨

n_total        = len(train)
n_never        = train['never_sold'].sum()
n_active       = n_total - n_never

# 활성 기간이 매우 짧은 제품 (30일 미만) 도 사실상 예측 불가
train['active_days'] = [
    (row['last_sale_date'] - row['first_sale_date']).days + 1
    if not row['never_sold'] else 0
    for _, row in train.iterrows()
]
n_very_sparse = ((train['active_days'] > 0) & (train['active_days'] < 30)).sum()

print('=== 판매 활성도별 제품 분류 ===')
print(f'  전체 제품 수                   : {n_total:,}개 (100%)')
print(f'  한 번도 안 팔린 제품           : {n_never:,}개 ({n_never/n_total:.1%})')
print(f'  활성기간 30일 미만 (희소)      : {n_very_sparse:,}개 ({n_very_sparse/n_total:.1%})')
print(f'  정상 활성 제품                 : {n_active - n_very_sparse:,}개 ({(n_active-n_very_sparse)/n_total:.1%})')
print()
print('💡 팀 논의 포인트:')
print('  -> 한 번도 안 팔린 제품은 제출 파일에서 그냥 0으로 처리?')
print('  -> 활성기간이 짧은 제품도 모델 학습에서 제외할지 결정 필요')

# 활성 기간 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 활성 기간 히스토그램
active_df = train[train['active_days'] > 0]
axes[0].hist(active_df['active_days'], bins=50, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].axvline(30, color='tomato', linestyle='--', label='30일 기준선')
axes[0].set_title('제품별 활성 기간 분포 (첫 판매 ~ 마지막 판매)', fontsize=12)
axes[0].set_xlabel('활성 기간 (일)')
axes[0].set_ylabel('제품 수')
axes[0].legend()
axes[0].grid(alpha=0.3)

# 제품 분류 파이 차트
labels = ['정상 활성', '희소 (30일 미만)', '미판매']
sizes  = [n_active - n_very_sparse, n_very_sparse, n_never]
colors = ['steelblue', 'orange', 'tomato']
axes[1].pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 11})
axes[1].set_title('제품 활성도 분류', fontsize=12)

plt.tight_layout()
plt.show()

# 학습에서 제외할 제품 ID 목록 저장 (나중에 STEP 2에서 사용)
NEVER_SOLD_IDS = set(train[train['never_sold']]['ID'].astype(str).str.zfill(5))
print(f'\n학습 제외 대상 (미판매): {len(NEVER_SOLD_IDS):,}개 제품')


============== 

밑에 step2 전까지는 다 지울수도 있음.(확인용)

**전체 판매 합계가 갑자기 튀는 날짜 13개를 찾음**


> 가능성 A: 그날 전체적으로 잘 팔린 것 (할인 행사, 명절 등)

> 가능성 B: 특정 제품 몇 개만 갑자기 튄 것

In [ ]:
import pandas as pd
import numpy as np

# ── 1. 전체 일별 합계 계산 ──────────────────────────────────────
daily_total = train[DATE_COLS].sum(axis=0)
daily_total.index = pd.to_datetime(daily_total.index)

# ── 2. MA30 트렌드 계산 ─────────────────────────────────────────
ma30 = daily_total.rolling(window=30, center=True, min_periods=1).mean()

# ── 3. 잔차 계산 ────────────────────────────────────────────────
residual = daily_total - ma30

# ── 4. spike 날짜 추출 (잔차 > 2σ) ─────────────────────────────
spike_dates = residual[residual > 2 * residual.std()].index
print(f"spike 날짜 수: {len(spike_dates)}개")
print(spike_dates.tolist())

In [ ]:
# ── 5. 제품별 기여도 분해 ───────────────────────────────────────
# DATE_COLS의 날짜 형식이 train 컬럼명과 정확히 일치해야 함
# (예: 'd_1', 'd_2' 형식이면 아래 strftime 대신 직접 매핑 필요)

product_mean = train[DATE_COLS].mean(axis=1)  # 제품별 전체 기간 평균

results = []
for date in spike_dates:
    # 날짜 컬럼명 형식에 맞게 수정 필요
    col = date.strftime('%Y-%m-%d')  # 컬럼명이 '2021-03-15' 형식일 때
    # col = f"d_{DATE_COLS.index(col) + 1}"  # 'd_숫자' 형식이면 이쪽

    if col not in train.columns:
        print(f"⚠️ {col} 컬럼 없음 — 컬럼 형식 확인 필요: {DATE_COLS[:3]}")
        break

    daily_row = train[col]
    ratio = daily_row / product_mean.replace(0, np.nan)

    top = ratio.sort_values(ascending=False).head(5)
    n_spiking = (ratio > 2).sum()  # 평소의 2배 이상 팔린 제품 수
    total_products = len(ratio)

    results.append({
        'date': date.date(),
        'n_spiking_products': n_spiking,
        'spike_ratio': n_spiking / total_products,  # 전체 중 몇 %가 튀었나
        'top_product_idx': top.index[0],
        'top_ratio': top.iloc[0],
    })

    print(f"\n{'='*50}")
    print(f"📅 {date.date()}  |  튄 제품: {n_spiking}/{total_products}개 ({n_spiking/total_products*100:.1f}%)")
    print(f"   상위 기여 제품 (평소 대비 배수):")
    for idx, val in top.items():
        print(f"   - 제품 {idx}: {val:.2f}배")

# ── 6. 판정 ─────────────────────────────────────────────────────
df_result = pd.DataFrame(results)
if len(df_result) > 0:
    avg_spike_ratio = df_result['spike_ratio'].mean()
    print(f"\n{'='*50}")
    print(f"📊 종합 판정")
    print(f"   spike 날짜당 평균 {avg_spike_ratio*100:.1f}%의 제품이 동시에 튐")

    if avg_spike_ratio > 0.3:
        print("   → 날짜 이벤트 (전체 영향) ✅  is_promo 더미 변수 추가 권장")
    elif avg_spike_ratio < 0.05:
        print("   → 특정 제품 이상치 ✅  해당 제품만 클리핑 또는 별도 처리")
    else:
        print("   → 혼재 ✅  날짜 더미 + 제품별 이상치 처리 병행 권장")

In [ ]:
# ── 심층 분석 ────────────────────────────────────────────────────

# 1. spike 날짜 간격 확인 (패턴이 있는지)
print("📅 spike 날짜 간격:")
for i in range(1, len(spike_dates)):
    gap = (spike_dates[i] - spike_dates[i-1]).days
    print(f"  {spike_dates[i-1].date()} → {spike_dates[i].date()} : {gap}일 간격")

# 2. spike 제품들이 같은 카테고리인지 확인
# (train에 카테고리 컬럼이 있다면)
print("\n\n📦 spike 제품 카테고리 분포:")
all_spike_products = []
for date in spike_dates:
    col = date.strftime('%Y-%m-%d')
    daily_row = train[col]
    ratio = daily_row / product_mean.replace(0, np.nan)
    # 2배 이상 튄 제품 인덱스 수집
    spiking = ratio[ratio > 2].index.tolist()
    all_spike_products.extend(spiking)

spike_product_series = pd.Series(all_spike_products)
print("가장 자주 등장한 spike 제품 (여러 날짜에 반복):")
print(spike_product_series.value_counts().head(20))
# → 같은 제품이 여러 날짜에 반복 등장하면 특정 제품 문제
# → 날짜마다 다른 제품이 튀면 날짜 이벤트

# 3. 459배짜리 제품 실제 판매 이력 확인 (데이터 오류 진단)
print("\n\n🔍 극단값 제품 판매 이력 (459배 제품):")
extreme_products = [15686, 15524, 4420]  # 위 결과에서 발견된 극단값 제품들
for pid in extreme_products:
    if pid in train.index:
        sales = train.loc[pid, DATE_COLS]
        nonzero = sales[sales > 0]
        print(f"\n  제품 {pid}:")
        print(f"    판매 있는 날: {len(nonzero)}일 / 전체 {len(DATE_COLS)}일")
        print(f"    최대값: {sales.max():.0f}  평균(0제외): {nonzero.mean():.2f}  중앙값: {nonzero.median():.2f}")
        print(f"    → ", end="")
        if len(nonzero) < 10:
            print("⚠️ 판매일이 극히 적음 — 평균이 0에 가까워서 비율이 뻥튀기된 것")
        else:
            print("✅ 판매 이력 충분 — 실제 spike 가능성")

# 4. spike 날짜 요일 확인
print("\n\n📆 spike 날짜 요일 분포:")
dow_map = {0:'월',1:'화',2:'수',3:'목',4:'금',5:'토',6:'일'}
for d in spike_dates:
    print(f"  {d.date()} ({dow_map[d.dayofweek]})")
# → 특정 요일에 몰리면 요일 집계 오류 의심

In [ ]:
repeat_spike_products = [1745, 11467, 12336, 12357, 11818]

for pid in repeat_spike_products:
    sales = train.loc[pid, DATE_COLS]  # 이 제품의 날짜별 판매량
    nonzero = sales[sales > 0]         # 판매가 0보다 많은 날만
    
    print(f"\n제품 {pid}")
    print(f"  전체 {len(DATE_COLS)}일 중 판매 있는 날: {len(nonzero)}일")
    print(f"  평균: {nonzero.mean():.1f}  최대: {nonzero.max():.0f}  중앙값: {nonzero.median():.1f}")
    
    # spike 날짜에 이 제품이 얼마나 팔렸는지만 확인
    spike_cols = [d.strftime('%Y-%m-%d') for d in spike_dates]
    spike_sales = sales[spike_cols]
    print(f"  spike 날짜 판매량:")
    print(f"  {spike_sales[spike_sales > 0].to_dict()}")

In [ ]:
# 전체 기간의 10% 이상 날에 팔린 제품만 "유효한 제품"으로 보기
min_sale_days = len(DATE_COLS) * 0.1  # 459일이면 약 46일 이상
valid_mask = train[DATE_COLS].gt(0).sum(axis=1) >= min_sale_days

print(f"유효 제품: {valid_mask.sum()}개")
print(f"제거 대상 희소 제품: {(~valid_mask).sum()}개")

# 이 필터 적용 후에 다시 spike 분석하면 훨씬 깔끔하게 나와요
train_valid = train[valid_mask]
daily_total_valid = train_valid[DATE_COLS].sum(axis=0)
daily_total_valid.index = pd.to_datetime(daily_total_valid.index)

>> 범인은 제품 12336 하나

제품 12336이 결정적인 이유:

중앙값이 1,052인데 spike 날에 200,420이 팔림 → 평소의 190배

전체 13개 spike 날짜에 전부 등장

이 제품 하나가 전체 일별 합계를 끌어올린 것

In [ ]:
# ── 제품 12336 정밀 해부 ─────────────────────────────────────────
pid = 12336
sales_12336 = train.loc[pid, DATE_COLS]
sales_12336.index = pd.to_datetime(DATE_COLS)

# 전체 판매 이력 시각화
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(16, 8))

# 위: 전체 이력
axes[0].plot(sales_12336.index, sales_12336.values, 
             color='#4c9be8', linewidth=0.8)
axes[0].axhline(sales_12336.median(), color='gold', 
                linestyle='--', linewidth=1.5, label=f'중앙값: {sales_12336.median():.0f}')
axes[0].axhline(sales_12336.mean(), color='tomato', 
                linestyle='--', linewidth=1.5, label=f'평균: {sales_12336.mean():.0f}')

# spike 날짜 강조
for d in spike_dates:
    axes[0].axvline(d, color='red', alpha=0.3, linewidth=2)

axes[0].set_title('제품 12336 전체 판매 이력 (빨간 수직선 = spike 날짜)', fontsize=13)
axes[0].legend()
axes[0].set_ylabel('판매량')

# 아래: 중앙값 3배 이상인 날만 확대
threshold = sales_12336.median() * 3
high_days = sales_12336[sales_12336 > threshold]
axes[1].bar(high_days.index, high_days.values, 
            color='tomato', alpha=0.8, width=2)
axes[1].axhline(threshold, color='gold', linestyle='--', 
                label=f'중앙값 3배 기준선 ({threshold:.0f})')
axes[1].set_title(f'제품 12336 — 중앙값 3배 초과 날짜 ({len(high_days)}일)', fontsize=13)
axes[1].legend()
axes[1].set_ylabel('판매량')

plt.tight_layout()
plt.show()

# ── 중복 기록 확인 ────────────────────────────────────────────────
print("연속 동일값 패턴 확인 (집계 오류 의심):")
for i in range(1, len(sales_12336)):
    if sales_12336.iloc[i] == sales_12336.iloc[i-1] and sales_12336.iloc[i] > threshold:
        print(f"  {sales_12336.index[i-1].date()} = {sales_12336.index[i].date()} "
              f"= {sales_12336.iloc[i]:.0f}  ← 중복 의심")

In [ ]:
# ── 처리 방법 ────────────────────────────────────────────────────

sales_12336 = train.loc[12336, DATE_COLS].copy()
sales_12336.index = pd.to_datetime(DATE_COLS)

# Step 1. 연속 동일값 → 뒷날을 0으로 교체 (중복 제거)
sales_fixed = sales_12336.copy()
for i in range(1, len(sales_fixed)):
    if sales_fixed.iloc[i] == sales_fixed.iloc[i-1] and sales_fixed.iloc[i] > 0:
        sales_fixed.iloc[i] = 0

print(f"중복 제거로 줄어든 총량: {sales_12336.sum() - sales_fixed.sum():,.0f}")

# Step 2. 그래도 남은 극단값 → 중앙값 기준 클리핑
median_val = sales_fixed[sales_fixed > 0].median()
cap = median_val * 20  # 중앙값의 20배를 상한선으로
sales_clipped = sales_fixed.clip(upper=cap)

print(f"클리핑 기준선(중앙값 20배): {cap:,.0f}")
print(f"클리핑으로 줄어든 총량: {sales_fixed.sum() - sales_clipped.sum():,.0f}")

# Step 3. 적용
train.loc[12336, DATE_COLS] = sales_clipped.values

# 전후 비교
print(f"\n처리 전 → 후")
print(f"  평균:   {sales_12336.mean():>10,.1f}  →  {sales_clipped.mean():>10,.1f}")
print(f"  중앙값: {sales_12336.median():>10,.1f}  →  {sales_clipped.median():>10,.1f}")
print(f"  최대값: {sales_12336.max():>10,.0f}  →  {sales_clipped.max():>10,.0f}")

# Step 4. 처리 후 전체 일별 합계 다시 계산해서 spike 사라졌는지 확인
daily_total_fixed = train[DATE_COLS].sum(axis=0)
daily_total_fixed.index = pd.to_datetime(DATE_COLS)
ma30_fixed = daily_total_fixed.rolling(30, center=True, min_periods=1).mean()
residual_fixed = daily_total_fixed - ma30_fixed
spike_after = residual_fixed[residual_fixed > 2 * residual_fixed.std()]

print(f"\nspike 날짜: 처리 전 13개 → 처리 후 {len(spike_after)}개")

In [ ]:
# spike 날짜가 같은지 다른지 확인
print("처리 후 spike 날짜:")
print(spike_after.index.tolist())

print("\n처리 전 spike 날짜:")
print(spike_dates.tolist())

# 새로 생긴 날짜 / 사라진 날짜
old_dates = set(spike_dates.date)
new_dates = set(spike_after.index.date)

print(f"\n사라진 날짜: {old_dates - new_dates}")
print(f"새로 생긴 날짜: {new_dates - old_dates}")
print(f"그대로인 날짜: {old_dates & new_dates}")

In [ ]:
# spike 날짜마다 "이 날 전체 합계를 가장 많이 올린 제품" 추출
# 기준: 그 날 판매량 - 그 제품의 평소 중앙값 = 초과 기여량

product_median = train[DATE_COLS].median(axis=1)  # 제품별 중앙값 (평균보다 robust)

top_contributors = []

for date in spike_dates:
    col = date.strftime('%Y-%m-%d')
    daily = train[col]
    
    # 초과 기여량 = 오늘 판매량 - 평소 중앙값
    excess = daily - product_median
    excess = excess[excess > 0]  # 평소보다 많이 팔린 제품만
    
    # 상위 5개
    top5 = excess.sort_values(ascending=False).head(5)
    
    for pid, val in top5.items():
        top_contributors.append({
            'date': date.date(),
            'product_id': pid,
            'excess': val,                          # 평소보다 얼마나 더 팔렸나
            'actual': daily[pid],                   # 실제 판매량
            'median': product_median[pid],          # 평소 중앙값
            'ratio': daily[pid] / (product_median[pid] + 1)  # 배수
        })

df_contrib = pd.DataFrame(top_contributors)

# 날짜별 상위 기여 제품 출력
print("=" * 60)
for date, grp in df_contrib.groupby('date'):
    total_excess = grp['excess'].sum()
    print(f"\n📅 {date}  (총 초과량: {total_excess:,.0f})")
    for _, row in grp.iterrows():
        print(f"   제품 {int(row['product_id']):>6}  "
              f"실제:{row['actual']:>10,.0f}  "
              f"평소중앙값:{row['median']:>8,.1f}  "
              f"초과:{row['excess']:>10,.0f}  "
              f"({row['ratio']:.1f}배)")

# 전체 날짜 통틀어 가장 많이 등장한 문제 제품
print("\n" + "=" * 60)
print("📦 전체 spike 날짜에 반복 등장한 제품 (누적 초과량 기준):")
summary = df_contrib.groupby('product_id')['excess'].agg(['sum','count']).sort_values('sum', ascending=False)
summary.columns = ['누적초과량', '등장횟수']
print(summary.head(15))

In [ ]:
# ── spike 관여 제품 전체를 한 번에 처리 ─────────────────────────

# 1. spike 날짜에 중앙값 대비 10배 이상 튄 제품 전체 수집
all_culprits = set()

for date in spike_dates:
    col = date.strftime('%Y-%m-%d')
    if col not in train.columns:
        continue
    daily_row = train[col]
    product_median = train[DATE_COLS].median(axis=1)
    ratio = daily_row / product_median.replace(0, np.nan)
    culprits = ratio[ratio > 10].index.tolist()
    all_culprits.update(culprits)

print(f"spike 관여 제품 수: {len(all_culprits)}개")
print(sorted(all_culprits))

# 2. 제품 번호 패턴 확인
culprit_ids = sorted(all_culprits)
print("\n제품 번호 분포:")
print(pd.Series(culprit_ids).describe())

# 3. 전부 동일하게 처리: 각 제품의 중앙값 * 20으로 클리핑
print(f"\n처리 전 전체 합계: {train[DATE_COLS].sum().sum():,.0f}")

for pid in all_culprits:
    product_sales = train.loc[pid, DATE_COLS]
    med = product_sales[product_sales > 0].median()
    if med > 0:
        cap = med * 20
        train.loc[pid, DATE_COLS] = product_sales.clip(upper=cap).values

print(f"처리 후 전체 합계: {train[DATE_COLS].sum().sum():,.0f}")

# 4. 처리 후 spike 재확인
daily_total_v2 = train[DATE_COLS].sum(axis=0)
daily_total_v2.index = pd.to_datetime(DATE_COLS)
ma30_v2 = daily_total_v2.rolling(30, center=True, min_periods=1).mean()
residual_v2 = daily_total_v2 - ma30_v2
spike_v2 = residual_v2[residual_v2 > 2 * residual_v2.std()]

print(f"\nspike 날짜: 처리 전 13개 → 처리 후 {len(spike_v2)}개")
if len(spike_v2) > 0:
    print("남은 spike 날짜:", spike_v2.index.date.tolist())

## STEP 2 — 데이터 전처리 & 피처 엔지니어링

### 파트 A: 데이터 형태 변환 (Wide -> Long)

현재 데이터 **(Wide 포맷)**:
```
ID    | 브랜드  | 2022-01-01 | 2022-01-02 | 2022-01-03 | ...
00000 | A브랜드 |     0      |     1      |     2      | ...
```

바꿀 데이터 포맷 **(Long 포맷)**:
```
ID    | 브랜드  | date       | qty
00000 | A브랜드 | 2022-01-01 |  0
00000 | A브랜드 | 2022-01-02 |  1
00000 | A브랜드 | 2022-01-03 |  2
```

**왜 Long 포맷이 필요한가?**  
Lag/Rolling 피처는 '날짜 순서'에 따라 계산하는데, Long 포맷이어야  
각 행이 '하나의 날짜 관측값'이 되어서 계산이 가능

---

### 파트 B: 피처 엔지니어링 (모델에게 더 많은 힌트 제공)

피처(Feature)란? -> 모델이 예측에 참고하는 **입력 정보**

| 피처 종류 | 예시 | 의미 |
|---|---|---|
| **Lag 피처** | qty_lag_7 | '7일 전에 얼마나 팔렸나?' |
| **Rolling 피처** | qty_roll_mean_28 | '최근 28일 평균 판매량은?' |
| **날짜 피처** | day_of_week, month | '오늘은 무슨 요일/월?' |
| **키워드 피처** | kw_log | '이 브랜드 검색량이 많아지고 있나?' |

---

### Log 변환이 필요한 이유

판매량 데이터의 문제점: 어떤 제품은 하루 1~2개, 어떤 제품은 수천 개  
-> 스케일 차이가 너무 커서 모델이 큰 값에만 집중하게 됩니다.

해결책: `log(x+1)` 변환으로 값의 차이를 균일하게 만듭니다.
```
원본:  0    1    10    100    1000
log1p: 0  0.69  2.40  4.61   6.91  (차이가 균일해짐)
```
나중에 `expm1()` 함수로 다시 원래 값으로 되돌릴 수 있습니다.


In [ ]:
# ============================================================
# 2-1. Wide -> Long 변환
# ============================================================
print('Wide -> Long 변환 중... (시간이 조금 걸립니다)')

# pd.melt(): Wide 포맷을 Long 포맷으로 변환하는 pandas 함수
# id_vars   : 그대로 유지할 컬럼 (제품 메타 정보)
# value_vars: 녹일 컬럼들 (날짜 컬럼들)
# var_name  : 날짜 컬럼명들이 들어갈 새 컬럼 이름
# value_name: 값들이 들어갈 새 컬럼 이름

qty_long = train.melt(
    id_vars=META_COLS,
    value_vars=DATE_COLS,
    var_name='date',
    value_name='qty'   # 판매 수량
)

# 매출액도 동일하게 변환
sales_long = sales.melt(
    id_vars=META_COLS,
    value_vars=DATE_COLS,
    var_name='date',
    value_name='revenue'  # 매출액
)

# 두 테이블을 같은 ID + 날짜 기준으로 합치기 (LEFT JOIN)
# merge: 두 DataFrame에서 공통 컬럼 기준으로 행을 연결
df = qty_long.merge(
    sales_long[['ID', 'date', 'revenue']],
    on=['ID', 'date'],
    how='left'  # 왼쪽(qty_long) 기준으로 합치기, 없으면 NaN
)

# 날짜 문자열 -> 실제 날짜 타입 (날짜 계산을 위해 필수)
df['date'] = pd.to_datetime(df['date'])

# 제품별, 날짜 오름차순 정렬 (Lag 계산시 순서가 중요!)
df = df.sort_values(['ID', 'date']).reset_index(drop=True)

print(f'변환 완료!')
print(f'  Wide: {train.shape}  ->  Long: {df.shape}')
print(f'  (행 수 = 제품 수 x 날짜 수 = {train.shape[0]:,} x {len(DATE_COLS)})')


In [ ]:
# ============================================================
# 2-2. 카테고리 인코딩 (텍스트 -> 숫자)
# ============================================================
# 딥러닝 모델은 숫자만 이해합니다.
# 'B002-C001-0002' 같은 텍스트는 직접 입력할 수 없으므로
# 각 고유 카테고리에 번호를 붙여줍니다.
# 예: '대분류B -> 0', '대분류C -> 1', '대분류D -> 2', ...

le_dict = {}  # 나중에 역변환이 필요할 때를 위해 인코더 저장

for col in ['대분류', '중분류', '소분류', '브랜드']:
    le = LabelEncoder()
    # fit_transform: 고유값 목록 학습(fit) + 변환(transform) 동시 수행
    df[col + '_enc'] = le.fit_transform(df[col].astype(str))
    le_dict[col] = le  # 인코더 저장 (나중에 역변환 가능)

print('카테고리 인코딩 완료')
for col in ['대분류', '중분류', '소분류', '브랜드']:
    print(f'  {col}: {df[col+"_enc"].nunique():,}개의 고유값')


In [ ]:
# ============================================================
# 2-3. 브랜드 키워드 검색량 병합
# ============================================================
# 사람들이 브랜드를 많이 검색 -> 관심 증가 -> 구매 증가 가능성
# 키워드 검색량은 판매량의 '선행 지표'가 될 수 있음

# keyword.csv도 Wide 포맷이므로 Long으로 변환
kw_long = keyword.melt(
    id_vars=['브랜드'],
    value_vars=[c for c in keyword.columns if c != '브랜드'],
    var_name='date',
    value_name='kw_cnt'  # 키워드 검색 횟수
)
kw_long['date'] = pd.to_datetime(kw_long['date'])

# 브랜드 + 날짜가 같은 행끼리 연결
df = df.merge(kw_long, on=['브랜드', 'date'], how='left')

# 키워드 데이터가 없는 경우 0으로 채우기
df['kw_cnt'] = df['kw_cnt'].fillna(0)
print(f'키워드 데이터 병합 완료. 현재 shape: {df.shape}')


### 📡 키워드 검색량이 판매의 선행 지표인가? — 유효성 검증

`brand_keyword_cnt.csv`는 **브랜드 단위**의 검색량입니다.
같은 브랜드의 모든 제품이 동일한 검색량 값을 공유하게 됩니다.

**검증 방법: 시차 상관관계 (Cross-Correlation)**

검색량이 N일 뒤의 판매량과 얼마나 상관관계가 있는지 측정합니다.
- lag=0: 검색량과 당일 판매량의 상관관계
- lag=7: 검색량과 7일 후 판매량의 상관관계

만약 lag=7~14일에서 상관계수가 높다면, 검색량이 판매량의 **7~14일 선행 지표**라는 의미입니다.

> **결과 해석 기준:**
> - |r| > 0.3: 약한 상관관계 (의미 있을 수 있음)
> - |r| > 0.5: 중간 상관관계 (피처로 유효)
> - |r| < 0.1: 거의 없음 (제거 검토)


In [ ]:
# ============================================================
# 2-3-1. 키워드 검색량 유효성 검증 (시차 상관관계 분석)
# ============================================================
# 앞서 병합한 df를 이용해 브랜드별 kw_cnt vs qty 의
# 시차(lag) 상관관계를 계산합니다.

# 브랜드별 일별 합산 데이터 준비
# (개별 제품이 아닌 브랜드 전체 판매량 vs 브랜드 검색량)
brand_daily = (
    df.groupby(['브랜드', 'date'])
    .agg(qty_sum=('qty', 'sum'), kw_cnt=('kw_cnt', 'first'))  # 브랜드별 일별 합산
    .reset_index()
    .sort_values(['브랜드', 'date'])
)

# 검색량이 있는 브랜드만 선택 (검색량 0인 브랜드는 분석 의미 없음)
active_brands = brand_daily.groupby('브랜드')['kw_cnt'].sum()
active_brands = active_brands[active_brands > 0].index.tolist()
print(f'검색량 데이터가 있는 브랜드: {len(active_brands)}개')

# 브랜드별 시차 상관관계 계산
lag_range = range(0, 15)  # 0일 ~ 14일 시차
corr_results = []

for brand in active_brands[:50]:  # 상위 50개 브랜드만 샘플링 (속도)
    brand_df = brand_daily[brand_daily['브랜드'] == brand].copy()
    if len(brand_df) < 30:  # 데이터가 너무 적으면 스킵
        continue
    for lag in lag_range:
        # shift(lag): kw_cnt를 lag일만큼 앞으로 당기기
        # 즉, 'lag일 전 검색량'이 '오늘 판매량'과 얼마나 연관되는지
        corr = brand_df['kw_cnt'].shift(lag).corr(brand_df['qty_sum'])
        corr_results.append({'brand': brand, 'lag': lag, 'corr': corr})

corr_df = pd.DataFrame(corr_results)

# lag별 평균 상관계수 계산
mean_corr = corr_df.groupby('lag')['corr'].mean()

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 평균 시차 상관계수
bars = axes[0].bar(mean_corr.index, mean_corr.values,
                   color=['tomato' if v == mean_corr.max() else 'steelblue'
                          for v in mean_corr.values],
                   alpha=0.8, edgecolor='white')
axes[0].axhline(0.3, color='orange', linestyle='--', alpha=0.7, label='r=0.3 기준선')
axes[0].set_title('검색량 -> 판매량 시차 상관관계 (lag별 평균)', fontsize=12)
axes[0].set_xlabel('lag (일) — 검색량이 몇 일 후 판매에 영향?')
axes[0].set_ylabel('평균 상관계수 (r)')
axes[0].legend()
axes[0].grid(alpha=0.3, axis='y')

# 상관계수 분포 (boxplot)
corr_pivot = corr_df.pivot_table(index='brand', columns='lag', values='corr')
axes[1].boxplot([corr_pivot[lag].dropna() for lag in lag_range],
                labels=list(lag_range), patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_title('lag별 브랜드간 상관계수 분포', fontsize=12)
axes[1].set_xlabel('lag (일)')
axes[1].set_ylabel('상관계수 (r)')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

best_lag = mean_corr.idxmax()
best_corr = mean_corr.max()
print(f'\n결과 요약:')
print(f'  가장 높은 상관관계: lag={best_lag}일  (r={best_corr:.3f})')
if best_corr > 0.3:
    print(f'  -> 키워드 검색량은 {best_lag}일 후 판매량과 상관관계가 있음 (피처로 유효)')
else:
    print(f'  -> 키워드 검색량의 선행 신호가 약함 (피처 유효성 낮을 수 있음)')
print(f'  -> 모델 학습 후 Feature Importance로 재검증 권장')


In [ ]:
# ============================================================
# 2-4. 날짜 피처 생성
# ============================================================
# 모델은 '2022-11-11'이라는 날짜 자체에서 의미를 모릅니다.
# '이 날은 11월이고, 금요일이고, 공휴일이다'라고 알려줘야 함.
# 날짜를 분해해서 각각의 숫자 피처로 만들어줍니다.

df['day_of_week']  = df['date'].dt.dayofweek    # 요일 (0=월 ~ 6=일)
df['day_of_month'] = df['date'].dt.day          # 날짜 (1~31)
df['month']        = df['date'].dt.month        # 월 (1~12)
df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)  # 연간 주차

# 주말이면 1, 평일이면 0 (온라인 구매 패턴이 주말에 다를 수 있음)
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

# 한국 주요 공휴일 (쇼핑 행동에 영향을 주는 날)
holidays_kr = pd.to_datetime([
    '2022-01-01', '2022-01-31', '2022-02-01', '2022-02-02',  # 신정, 설날
    '2022-03-01', '2022-05-05', '2022-06-06', '2022-08-15',  # 삼일절, 어린이날, 현충일, 광복절
    '2022-09-09', '2022-09-10', '2022-09-11', '2022-09-12',  # 추석 연휴
    '2022-10-03', '2022-10-10', '2022-12-25',                # 개천절, 한글날, 크리스마스
    '2023-01-01', '2023-01-21', '2023-01-22', '2023-01-23', '2023-01-24',  # 신정, 설날
    '2023-03-01', '2023-05-05', '2023-06-06',                # 삼일절, 어린이날, 현충일
])
df['is_holiday'] = df['date'].isin(holidays_kr).astype(int)  # 공휴일: 1, 평일: 0

print('날짜 피처 생성 완료')
print('  생성: day_of_week, day_of_month, month, week_of_year, is_weekend, is_holiday')


In [ ]:
# ============================================================
# 2-5. Log 변환 (스케일 안정화)
# ============================================================
# np.log1p(x) = log(x+1)
# x=0일 때도 log(0+1)=0이 되어 정상 처리됨
# 나중에 np.expm1(y) = exp(y)-1 으로 역변환 가능

df['qty_log']     = np.log1p(df['qty'])               # 판매 수량
df['revenue_log'] = np.log1p(df['revenue'].fillna(0)) # 매출액 (결측치 0처리)
df['kw_log']      = np.log1p(df['kw_cnt'])            # 키워드 검색량

print('Log 변환 완료')
print('변환 예시 (판매수량):')
for v in [0, 1, 10, 100, 1000]:
    print(f'  원본: {v:5d}  ->  log 변환: {np.log1p(v):.3f}  ->  역변환: {np.expm1(np.log1p(v)):.1f}')


### 📦 product_info.csv — 텍스트 키워드 피처 추출

`product_info.csv`에는 제품 특성이 텍스트로 담겨 있습니다.
예: `'제품유형:일반식품 콜라겐 펩타이드:1000mg 섭취대상:성인남녀 제품타입:분말'`

이 텍스트에서 **판매 패턴에 영향을 줄 수 있는 키워드를 이진 피처(0/1)로 추출**합니다.

**왜 텍스트 그대로가 아닌 키워드 추출인가?**

딥러닝 모델은 텍스트를 직접 이해하지 못합니다. BERT 같은 임베딩도 가능하지만
지금 일정에서는 키워드 기반 이진 피처가 가장 현실적이고 해석도 쉽습니다.

**추출 전략:**
- 건강 기능성 키워드: 체지방감소, 콜라겐, 비타민 등 (시즌성 구매 영향)
- 제품 형태: 분말, 정, 액상 등 (재구매 주기 차이)
- 섭취 기간: 1개월분, 3개월분 등 (구매 주기 예측에 활용 가능)


In [ ]:
# ============================================================
# 2-5-1. product_info.csv 텍스트 키워드 피처 추출
# ============================================================

print('product_info.csv 키워드 피처 추출 중...')
print(f'product 데이터 형태: {product.shape}')
print(product.head(3))
print()

# ── 추출할 키워드 그룹 정의 ─────────────────────────────────
# 판매량에 영향을 줄 것으로 예상되는 키워드들을 카테고리별로 묶음
KEYWORD_GROUPS = {
    # 건강 기능성 (시즌성 구매 패턴 차이: 다이어트 제품은 1월/여름에 급증)
    'kw_diet':      ['체지방', '다이어트', '칼로리'],
    'kw_collagen':  ['콜라겐', '피부'],
    'kw_vitamin':   ['비타민', '미네랄', '아연', '철분'],
    'kw_omega':     ['오메가', '오일', '피쉬오일'],
    'kw_probiotic': ['유산균', '프로바이오틱'],
    'kw_protein':   ['단백질', '프로틴'],
    'kw_immune':    ['면역', '항산화'],

    # 제품 형태 (형태에 따라 재구매 주기 다름)
    'kw_powder':    ['분말', '파우더'],
    'kw_tablet':    [':정', '정제', '캡슐'],
    'kw_liquid':    ['액상', '음료'],
    'kw_stick':     ['스틱', '포'],

    # 섭취 기간 (구매 주기 예측에 활용 가능)
    'kw_1month':    ['1개월분', '30일분'],
    'kw_3month':    ['3개월분', '90일분'],

    # 섭취 대상
    'kw_child':     ['어린이', '키즈', '아동'],
    'kw_woman':     ['여성', '임산부', '갱년기'],
    'kw_senior':    ['노인', '시니어', '50대', '60대'],
}

# ── 각 제품에 대해 키워드 포함 여부 (0/1) 계산 ──────────────

# product_info 데이터 정리
# '제품' 컬럼이 train의 '제품' 컬럼과 연결되는 키
product_feat = product[['제품']].copy()

# 텍스트 컬럼 통합 (여러 컬럼이 있을 경우 모두 합치기)
text_cols = [c for c in product.columns if c != '제품']
product_feat['text_all'] = product[text_cols].fillna('').astype(str).apply(
    lambda row: ' '.join(row), axis=1
).str.lower()  # 소문자로 통일

# 키워드별 이진 피처 생성
for feat_name, keywords in KEYWORD_GROUPS.items():
    # 키워드 중 하나라도 포함되면 1, 아니면 0
    pattern = '|'.join(keywords)  # 'OR' 조건으로 합치기
    product_feat[feat_name] = product_feat['text_all'].str.contains(
        pattern, na=False
    ).astype(int)

product_feat = product_feat.drop(columns=['text_all'])
PRODUCT_KW_COLS = [c for c in product_feat.columns if c.startswith('kw_')]

print(f'\n추출된 키워드 피처: {len(PRODUCT_KW_COLS)}개')
for col in PRODUCT_KW_COLS:
    cnt = product_feat[col].sum()
    print(f'  {col:20s}: {cnt:4d}개 제품 ({cnt/len(product_feat):.1%})')


In [ ]:
# ============================================================
# 2-5-2. product_info 키워드 피처를 df에 병합
# ============================================================
# train의 '제품' 컬럼과 product_feat의 '제품' 컬럼을 key로 연결

df = df.merge(
    product_feat,
    on='제품',   # train의 제품 코드와 product_info의 제품 코드 매칭
    how='left'  # 매칭 안 되는 제품은 NaN -> 0으로 처리
)

# 매칭 안 된 제품(신제품 등)은 0으로 채우기
df[PRODUCT_KW_COLS] = df[PRODUCT_KW_COLS].fillna(0).astype(int)

print(f'키워드 피처 병합 완료. 현재 df shape: {df.shape}')
print(f'매칭 안 된 제품 수: {df[PRODUCT_KW_COLS[0]].isna().sum()}개')


In [ ]:
# ============================================================
# 2-6. Lag & Rolling 피처 생성 (가장 중요한 피처들!)
# ============================================================
#
# -- Lag 피처란? -----------------------------------------
# 'N일 전에 얼마나 팔렸는가?'
#   qty_lag_1  = 어제 판매량       (단기 트렌드)
#   qty_lag_7  = 7일 전 판매량     (지난주 같은 요일 = 주간 패턴)
#   qty_lag_14 = 14일 전 판매량    (2주 전)
#   qty_lag_28 = 28일 전 판매량    (4주 전 = 월간 패턴)
#
# -- Rolling 피처란? -------------------------------------
# '최근 N일 동안의 평균/변동성은?'
#   qty_roll_mean_7  = 최근 7일 평균  (단기 추세)
#   qty_roll_mean_28 = 최근 28일 평균 (중기 추세)
#   qty_roll_std_7   = 최근 7일 표준편차 (변동성)
#
# ⚠️ 중요: shift(1)을 쓰는 이유
# 오늘 판매량으로 오늘을 예측 = '미래 정보 누출(Data Leakage)'!
# 반드시 1일 뒤로 밀어서 어제까지의 정보만 사용해야 함.

print('Lag & Rolling 피처 생성 중... (약 1~2분 소요)')

# 제품(ID)별로 독립적으로 계산해야 함 (다른 제품 데이터가 섞이면 안됨)
df = df.sort_values(['ID', 'date']).reset_index(drop=True)

# -- 판매 수량 Lag 피처 ----------------------------------
for lag in [1, 7, 14, 28]:
    # shift(lag): lag만큼 아래로 밀기 = lag일 전 값이 현재 행에 옴
    df[f'qty_lag_{lag}'] = df.groupby('ID')['qty_log'].shift(lag)
    print(f'  qty_lag_{lag} 생성 완료')

# -- 판매 수량 Rolling 피처 ------------------------------
for window in [7, 14, 28]:
    grp = df.groupby('ID')['qty_log']
    # shift(1): 오늘 값 제외 후 -> rolling(window): 최근 N일 계산
    df[f'qty_roll_mean_{window}'] = grp.transform(
        lambda x: x.shift(1).rolling(window, min_periods=1).mean()
    )
    df[f'qty_roll_std_{window}'] = grp.transform(
        lambda x: x.shift(1).rolling(window, min_periods=1).std().fillna(0)
    )
    print(f'  qty_roll_mean_{window}, qty_roll_std_{window} 생성 완료')

# -- 키워드 검색량 Lag & Rolling -------------------------
for lag in [7, 14]:
    df[f'kw_lag_{lag}'] = df.groupby('ID')['kw_log'].shift(lag)
df['kw_roll_mean_7'] = df.groupby('ID')['kw_log'].transform(
    lambda x: x.shift(1).rolling(7, min_periods=1).mean()
)

# 제품 초반 날짜들은 lag 계산 불가 -> NaN -> 0으로 채우기
lag_roll_cols = [c for c in df.columns if 'lag' in c or 'roll' in c]
df[lag_roll_cols] = df[lag_roll_cols].fillna(0)

print(f'\n피처 생성 완료! 총 컬럼 수: {df.shape[1]}개')


In [ ]:
# ============================================================
# 2-7. 최종 피처 목록 정의 (product_info 키워드 피처 추가)
# ============================================================

FEATURE_COLS = [
    # ── 카테고리 ────────────────────────────────────────
    '대분류_enc', '중분류_enc', '소분류_enc', '브랜드_enc',

    # ── 날짜 피처 ───────────────────────────────────────
    'day_of_week', 'day_of_month', 'month', 'week_of_year',
    'is_weekend', 'is_holiday',

    # ── 과거 판매량 Lag ──────────────────────────────────
    'qty_lag_1', 'qty_lag_7', 'qty_lag_14', 'qty_lag_28',

    # ── 이동 평균 & 표준편차 ─────────────────────────────
    'qty_roll_mean_7', 'qty_roll_mean_14', 'qty_roll_mean_28',
    'qty_roll_std_7',  'qty_roll_std_14',  'qty_roll_std_28',

    # ── 키워드 검색량 (선행 지표) ────────────────────────
    'kw_log', 'kw_lag_7', 'kw_lag_14', 'kw_roll_mean_7',

    # ── 매출액 ───────────────────────────────────────────
    'revenue_log',

    # ── product_info 텍스트 키워드 피처 (NEW) ─────────────
    # 제품 특성에 따른 판매 패턴 차이를 반영
    # 예: 다이어트 제품은 1월/여름에 판매 급증 패턴
    'kw_diet', 'kw_collagen', 'kw_vitamin', 'kw_omega',
    'kw_probiotic', 'kw_protein', 'kw_immune',
    'kw_powder', 'kw_tablet', 'kw_liquid', 'kw_stick',
    'kw_1month', 'kw_3month',
    'kw_child', 'kw_woman', 'kw_senior',
]

TARGET_COL = 'qty_log'

# PRODUCT_KW_COLS에서 df에 없는 피처 제거 (안전장치)
FEATURE_COLS = [f for f in FEATURE_COLS if f in df.columns]

print(f'총 피처 수: {len(FEATURE_COLS)}개')
print(f'  카테고리 피처       : 4개')
print(f'  날짜 피처           : 6개')
print(f'  Lag/Rolling 피처    : 10개')
print(f'  키워드 검색량 피처  : 4개')
print(f'  매출액 피처         : 1개')
print(f'  product_info 피처   : {len([f for f in FEATURE_COLS if f.startswith("kw_") and f not in ["kw_log","kw_lag_7","kw_lag_14","kw_roll_mean_7"]])}개')


In [ ]:
# ============================================================
# 2-8. 피처 스케일링 (MinMaxScaler)
# ============================================================
# 피처들의 크기(스케일)가 서로 다름:
#   month: 1~12,  day_of_month: 1~31,  qty_roll_mean_28: 0~수백
# -> 큰 숫자의 피처에만 모델이 집중하는 문제 발생
# -> MinMaxScaler로 모든 피처를 0~1 범위로 균일화
#
# ⚠️ 중요 규칙: fit은 '학습 데이터'에만!
# 검증/테스트 데이터에 fit하면 '미래 정보 누출' 발생

scaler = MinMaxScaler()

# 학습 데이터만 선택 (예측 시작일 이전)
train_mask = df['date'] < pd.to_datetime(PRED_DATES[0])

# 학습 데이터로만 scaler 훈련(fit)
scaler.fit(df[train_mask][FEATURE_COLS])

# 전체 데이터에 변환(transform) 적용
df[FEATURE_COLS] = scaler.transform(df[FEATURE_COLS])

print('MinMaxScaler 스케일링 완료 (0~1 범위로 정규화)')
print(f'  fit 사용 기간: ~ {PRED_DATES[0]} 이전 데이터')
print(f'  transform 적용: 전체 {len(df):,}행')


## STEP 3 — 딥러닝 모델 구성 (PyTorch LSTM)

### 슬라이딩 윈도우란? (Dataset의 핵심 개념)

데이터를 모델에 넣기 위해 **슬라이딩 윈도우(Sliding Window)** 방식을 사용합니다.
긴 시계열에서 (입력구간 -> 출력구간) 쌍을 반복해서 추출하는 방식입니다:

```
전체 시계열: [d1, d2, d3, ..., d459]

샘플 1: 입력 [d1~d28]  -> 정답 [d29~d49]
샘플 2: 입력 [d2~d29]  -> 정답 [d30~d50]
샘플 3: 입력 [d3~d30]  -> 정답 [d31~d51]
...
```

이렇게 하면 하나의 제품 시계열에서 수백 개의 학습 샘플을 만들 수 있습니다.

---

### LSTM 아키텍처 설명

```
입력 (배치 크기 x 28일 x 25개 피처)
    |
    v
Bidirectional LSTM 1층  (각 방향 hidden=128, 양방향 합계 출력=256)
    |
    v
Bidirectional LSTM 2층  (동일)
    |
    v
Attention (28개 시점 중 어떤 날이 중요한지 가중치 자동 학습)
    |
    v
FC Layer  (256 -> 128 -> 64 -> 21)
    |
    v
출력: 미래 21일 예측값
```

**Bidirectional이란?** 앞-뒤 방향으로만 읽는 것이 아니라 뒤-앞 방향도 읽어서 맥락을 더 풍부하게 이해합니다.

**Attention이란?** '28일 중 어떤 시점의 정보가 예측에 더 중요한지' 가중치를 스스로 학습합니다.


In [ ]:
# ============================================================
# 3-1. 하이퍼파라미터 설정
# ============================================================
# 하이퍼파라미터: 모델 학습 전에 사람이 직접 정해주는 설정값들
# (모델이 학습하는 가중치와 달리, 사람이 조정해야 함)

SEQ_LEN    = 28     # 입력 시퀀스 길이: 과거 몇 일을 볼 것인가?
                    # 28일 = 4주 -> 주간 패턴(7일 주기)을 4번 반복해서 볼 수 있음
PRED_LEN   = 21     # 출력 길이: 미래 몇 일을 예측할 것인가? (제출 요구사항)
N_FEAT     = len(FEATURE_COLS)  # 피처 수 (자동 계산)

HIDDEN     = 128    # LSTM 은닉층 크기: 클수록 복잡한 패턴 학습 가능 (but 느려짐)
N_LAYERS   = 2      # LSTM 레이어 수: 깊을수록 더 추상적인 패턴 학습
DROPOUT    = 0.3    # 학습 중 30%의 뉴런을 랜덤으로 끔 -> 과적합 방지

BATCH_SIZE = 512    # 한 번에 학습할 샘플 수 (GPU 메모리에 맞게 조절)
LR         = 1e-3   # 학습률: 너무 크면 발산, 너무 작으면 학습이 느림
EPOCHS     = 100    # 최대 학습 반복 횟수
PATIENCE   = 10     # 검증 성능이 10 epoch 동안 개선 없으면 학습 중단
VAL_DAYS   = 30     # 검증에 사용할 마지막 N일 (학습에 사용 안 함)

print('=== 하이퍼파라미터 설정 ===')
print(f'  입력 시퀀스 길이 : {SEQ_LEN}일')
print(f'  예측 길이        : {PRED_LEN}일')
print(f'  피처 수          : {N_FEAT}개')
print(f'  LSTM 은닉 크기   : {HIDDEN}')
print(f'  LSTM 레이어 수   : {N_LAYERS}')
print(f'  드롭아웃 비율    : {DROPOUT}')
print(f'  배치 크기        : {BATCH_SIZE}')
print(f'  학습률           : {LR}')


In [ ]:
# ============================================================
# 3-2. Dataset 클래스 정의
# ============================================================
# PyTorch에서는 데이터를 Dataset 클래스로 포장해야 함.
# 두 가지 메서드를 반드시 구현:
#   __len__     : 전체 샘플 수 반환
#   __getitem__ : 특정 번호(idx)의 샘플 반환

class SalesDataset(Dataset):
    # 슬라이딩 윈도우 방식으로 학습 샘플을 생성하는 Dataset
    # 각 샘플:
    #   X(입력): shape=(SEQ_LEN, N_FEAT) = (28, 25) <- 과거 28일 피처
    #   y(정답): shape=(PRED_LEN,) = (21,)           <- 미래 21일 판매량

    def __init__(self, df, feature_cols, target_col, seq_len, pred_len):
        self.samples = []  # (X, y) 쌍을 저장할 리스트

        # 제품(ID)별로 독립적으로 슬라이딩 윈도우 적용
        # 서로 다른 제품의 날짜가 연결되면 안 되므로 groupby 필수!
        for pid, grp in df.groupby('ID'):
            grp  = grp.sort_values('date').reset_index(drop=True)
            feat = grp[feature_cols].values.astype(np.float32)  # 피처 배열
            tgt  = grp[target_col].values.astype(np.float32)    # 정답 배열
            n    = len(grp)  # 이 제품의 총 날짜 수

            # 슬라이딩 윈도우: i를 한 칸씩 이동
            for i in range(n - seq_len - pred_len + 1):
                x = feat[i : i + seq_len]                         # 입력: 28일
                y = tgt[i + seq_len : i + seq_len + pred_len]     # 정답: 21일
                self.samples.append((x, y))

    def __len__(self):
        return len(self.samples)  # 전체 샘플 수

    def __getitem__(self, idx):
        x, y = self.samples[idx]
        # numpy 배열 -> PyTorch Tensor 변환 (GPU 연산에 필요)
        return torch.tensor(x), torch.tensor(y)


# ============================================================
# 3-3. Train / Validation 시간 기준 분리
# ============================================================
# 시계열 데이터는 반드시 시간 순서를 지켜서 분리해야 함!
# (랜덤 분리 시 미래 데이터로 과거를 예측하는 오류 발생)

cut_date = pd.to_datetime(DATE_COLS[-VAL_DAYS])  # 검증 시작일

df_tr  = df[df['date'] <  cut_date]  # 학습: cut_date 이전
df_val = df[df['date'] >= cut_date]  # 검증: cut_date 이후 (마지막 30일)

print(f'학습 기간: {df_tr["date"].min().date()}  ~  {df_tr["date"].max().date()}')
print(f'검증 기간: {df_val["date"].min().date()}  ~  {df_val["date"].max().date()}')

train_ds = SalesDataset(df_tr,  FEATURE_COLS, TARGET_COL, SEQ_LEN, PRED_LEN)
val_ds   = SalesDataset(df_val, FEATURE_COLS, TARGET_COL, SEQ_LEN, PRED_LEN)

# DataLoader: Dataset을 배치 단위로 잘라서 공급
# shuffle=True  -> 학습 시 샘플 순서 섞기 (과적합 방지)
# num_workers=4 -> 데이터 로딩에 4개 CPU 코어 병렬 사용
# pin_memory=True -> GPU 전송 속도 향상
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f'\n학습 샘플 수: {len(train_ds):,}개')
print(f'검증 샘플 수: {len(val_ds):,}개')


In [ ]:
# ============================================================
# 3-4. LSTM 모델 클래스 정의
# ============================================================

class SalesLSTM(nn.Module):
    # 판매량 예측을 위한 Bidirectional LSTM + Attention 모델

    def __init__(self, n_features, hidden_size, n_layers, pred_len, dropout=0.3):
        # 부모 클래스 초기화 (PyTorch 모델 작성 시 항상 필요)
        super().__init__()
        self.hidden_size = hidden_size

        # ── LSTM 레이어 ─────────────────────────────────────
        # input_size   : 각 시점의 피처 수 (25)
        # hidden_size  : 기억 용량 (128)
        # num_layers   : LSTM 몇 층 쌓을지 (2)
        # batch_first  : True -> 입력 형태 (batch, seq, feature)
        # dropout      : 레이어 간 드롭아웃 (과적합 방지)
        # bidirectional: True -> 양방향 처리 -> 출력 크기 = hidden*2
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0,
            bidirectional=True
        )

        # ── Attention 레이어 ─────────────────────────────────
        # 28개 시점 각각의 중요도 점수를 1개 숫자로 계산
        self.attn = nn.Linear(hidden_size * 2, 1)

        # ── FC Head (최종 예측값 출력) ───────────────────────
        # 점진적으로 차원을 줄여가며 최종 21개 예측값 출력
        self.head = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),  # 256 -> 128
            nn.ReLU(),          # 비선형 활성화 함수 (복잡한 패턴 학습)
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2), # 128 -> 64
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, pred_len),    # 64 -> 21 (최종 출력!)
        )

    def forward(self, x):
        # x shape: (batch_size, seq_len, n_features)

        # 1) LSTM 통과: 각 시점의 출력 벡터 계산
        out, _ = self.lstm(x)
        # out shape: (batch, seq_len=28, hidden*2=256)

        # 2) Attention: 28개 시점 중 어떤 시점이 중요한지 가중치 계산
        attn_scores  = self.attn(out)                     # (batch, 28, 1)
        attn_weights = torch.softmax(attn_scores, dim=1)  # 합이 1이 되도록 정규화
        context      = (attn_weights * out).sum(dim=1)    # 가중합 -> (batch, 256)

        # 3) FC Head: 최종 21일 예측값 계산
        return self.head(context)  # (batch, 21)


# 모델 인스턴스 생성 후 GPU로 이동
# .to(DEVICE): GPU가 있으면 GPU 메모리에 모델 올리기
model = SalesLSTM(
    n_features=N_FEAT,
    hidden_size=HIDDEN,
    n_layers=N_LAYERS,
    pred_len=PRED_LEN,
    dropout=DROPOUT
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'\n총 학습 가능 파라미터 수: {total_params:,}개')


## STEP 4 — 컴파일 (MAE Loss, Adam Optimizer)

### 손실 함수(Loss Function)란?

모델의 예측값과 실제값의 차이를 수치로 표현하는 함수입니다.  
**학습의 목표 = 이 값을 최대한 줄이는 것**입니다.

**왜 MAE를 사용하나요?**
- MAE = 예측값과 실제값의 절대 차이의 평균 (예: 실제 10개 예측 7개 -> 오차 3)
- 직관적이고, 이상치(극단값)에 덜 민감
- 대회 평가지표가 MAE이므로 손실 함수도 MAE로 맞추는 것이 유리

### 옵티마이저(Optimizer)란?

손실을 줄이기 위해 모델의 가중치를 어떻게 업데이트할지 결정하는 알고리즘입니다.

**Adam을 사용하는 이유:**  
- 파라미터마다 학습률을 자동으로 조절 -> 빠르고 안정적  
- 딥러닝에서 가장 많이 쓰이는 기본 옵티마이저


In [ ]:
# ============================================================
# 손실 함수 & 옵티마이저 설정
# ============================================================

# MAE Loss: |예측값 - 실제값|의 평균
criterion = nn.L1Loss()

# Adam 옵티마이저
# weight_decay: L2 정규화 -> 가중치가 너무 커지는 것 방지 -> 과적합 방지
optimizer = torch.optim.Adam(
    model.parameters(),  # 학습할 파라미터 목록
    lr=LR,               # 학습률
    weight_decay=1e-5    # L2 정규화 계수
)

print('손실 함수: MAE (nn.L1Loss)')
print('옵티마이저: Adam')
print(f'  학습률(lr): {LR}')
print(f'  weight_decay: 1e-5')


## STEP 5 — Callbacks 정의 (EarlyStopping, ReduceLROnPlateau)

### Callback이란?

학습 중 특정 조건이 되면 자동으로 실행되는 함수들입니다.  
학습을 자동으로 관리해서 사람이 일일이 모니터링할 필요가 없습니다.

### EarlyStopping (조기 종료)

검증 성능이 N번 이상 개선되지 않으면 학습을 자동으로 중단합니다.
- **과적합(Overfitting) 방지**: 학습 데이터에만 너무 잘 맞고 새 데이터에는 못하는 상황
- 최고 성능일 때의 모델을 파일로 저장해두고, 학습 종료 후 불러옴

### ReduceLROnPlateau (학습률 자동 감소)

검증 성능 개선이 멈추면 학습률을 절반으로 줄입니다.
- 학습 후반부에 더 세밀하게 가중치를 조정할 수 있음
- 마치 목적지 가까이 갈수록 발걸음을 줄이는 것과 같음


In [ ]:
# ============================================================
# 5-1. ReduceLROnPlateau 스케줄러
# ============================================================

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',      # 'min': 손실(loss)이 줄어드는 방향이 좋음
    factor=0.5,      # 학습률을 현재의 50%로 줄임 (예: 0.001 -> 0.0005)
    patience=5,      # 5 epoch 동안 개선 없으면 학습률 감소
    verbose=True     # 학습률 변경 시 메시지 출력
)

# ============================================================
# 5-2. EarlyStopping 클래스 정의
# ============================================================

class EarlyStopping:
    # 검증 손실이 개선되지 않으면 학습을 조기 종료하는 클래스

    def __init__(self, patience=10, delta=1e-5, path='best_model.pt'):
        self.patience   = patience    # 몇 epoch 동안 참을지
        self.delta      = delta       # 이 값보다 적게 개선되면 '개선 없음'으로 간주
        self.path       = path        # 최고 모델 저장 경로
        self.best_loss  = np.inf      # 지금까지의 최고 검증 손실 (처음엔 무한대)
        self.counter    = 0           # 개선 없는 epoch 카운터
        self.early_stop = False       # 학습 중단 신호

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.delta:
            # 개선됨: 최고 기록 갱신 & 모델 저장
            self.best_loss = val_loss
            self.counter   = 0
            # state_dict(): 모델의 가중치(파라미터)만 저장 (모델 구조 제외, 용량 절약)
            torch.save(model.state_dict(), self.path)
        else:
            # 개선 없음: 카운터 증가
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True  # 학습 중단 신호 발동!


MODEL_PATH    = os.path.join(OUTPUT_DIR, 'best_model.pt')
early_stopper = EarlyStopping(patience=PATIENCE, path=MODEL_PATH)

print('Callback 설정 완료')
print(f'  EarlyStopping    : {PATIENCE} epoch 동안 개선 없으면 중단')
print(f'  ReduceLROnPlateau: 5 epoch 동안 개선 없으면 LR x 0.5')
print(f'  최적 모델 저장   : {MODEL_PATH}')


## STEP 6 — 모델 학습 및 검증

### 학습 루프가 동작하는 방식

딥러닝 학습은 아래 과정을 수백~수천 번 반복합니다:

```
[1 Epoch = 전체 학습 데이터를 한 번 다 봄]

  배치(Batch) 반복 ─────────────────────────────────────────┐
  1. 입력(X)을 모델에 넣어 예측값 계산  (Forward Pass)      │
  2. 예측값과 정답의 차이(Loss) 계산                        │
  3. Loss 기반으로 기울기(Gradient) 계산  (Backward Pass)   │
  4. 기울기 방향으로 가중치 업데이트  (Optimizer Step)      │
  ────────────────────────────────────────────────────────────

  검증: 검증 데이터로 현재 성능 측정 (가중치 업데이트 없이!)
  Callback: EarlyStopping, ReduceLR 조건 체크
```

### Gradient Clipping이란?

기울기(gradient)가 너무 커지면 학습이 불안정해지는 '폭발' 현상이 생깁니다.  
`clip_grad_norm_`으로 기울기의 최대 크기를 제한해서 안정적인 학습을 유지합니다.


In [ ]:
# ============================================================
# 6-1. 1 Epoch 학습 함수
# ============================================================

def train_epoch(model, loader, criterion, optimizer):
    # 학습 데이터 전체를 한 번 돌며 모델 가중치를 업데이트하는 함수
    # 반환값: 이번 epoch의 평균 MAE Loss

    model.train()   # 학습 모드 (Dropout이 랜덤으로 작동)
    total_loss = 0.0

    for X, y in loader:
        # 1) 데이터를 GPU로 이동 (모델과 같은 디바이스에 있어야 함)
        X, y = X.to(DEVICE), y.to(DEVICE)

        # 2) 기울기 초기화 (이전 배치의 기울기가 누적되지 않도록)
        optimizer.zero_grad()

        # 3) 순전파: 입력 -> 예측
        pred = model(X)  # shape: (batch, 21)

        # 4) 손실 계산: 예측값 vs 실제값
        loss = criterion(pred, y)

        # 5) 역전파: Loss를 각 가중치로 미분 (기울기 계산)
        loss.backward()

        # 6) Gradient Clipping: 기울기 폭발 방지 (최대 norm = 1.0)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # 7) 가중치 업데이트: 기울기 방향으로 한 걸음 이동
        optimizer.step()

        total_loss += loss.item() * len(X)

    return total_loss / len(loader.dataset)  # 평균 loss


# ============================================================
# 6-2. 1 Epoch 검증 함수
# ============================================================

def eval_epoch(model, loader, criterion):
    # 검증 데이터로 현재 모델 성능을 측정하는 함수
    # 가중치 업데이트 없음! (측정만 수행)

    model.eval()    # 평가 모드 (Dropout 비활성화 -> 모든 뉴런 사용)
    total_loss = 0.0

    with torch.no_grad():  # 기울기 계산 비활성화 -> 메모리 절약, 속도 향상
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            pred = model(X)
            loss = criterion(pred, y)
            total_loss += loss.item() * len(X)

    return total_loss / len(loader.dataset)


In [ ]:
# ============================================================
# 6-3. 메인 학습 루프
# ============================================================

# 학습 기록 저장 딕셔너리 (나중에 그래프로 그릴 것)
history = {
    'train_loss': [],  # epoch별 학습 손실
    'val_loss':   [],  # epoch별 검증 손실
    'lr':         [],  # epoch별 학습률
}

print(f'학습 시작! | 디바이스: {DEVICE} | 최대 Epoch: {EPOCHS}')
print('=' * 62)

for epoch in tqdm(range(1, EPOCHS + 1), desc='Training'):

    # 1) 학습 & 검증 수행
    tr_loss  = train_epoch(model, train_loader, criterion, optimizer)
    val_loss = eval_epoch(model, val_loader, criterion)

    # 2) Callback 업데이트
    scheduler.step(val_loss)       # 검증 손실 기반으로 LR 조절
    early_stopper(val_loss, model) # 최고 모델 저장 & 중단 여부 판단

    # 3) 기록 저장
    current_lr = optimizer.param_groups[0]['lr']  # 현재 학습률
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(val_loss)
    history['lr'].append(current_lr)

    # 4) 로그 출력 (5 epoch마다)
    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS}  |  '
              f'Train MAE: {tr_loss:.5f}  |  '
              f'Val MAE: {val_loss:.5f}  |  '
              f'LR: {current_lr:.2e}')

    # 5) 조기 종료 체크
    if early_stopper.early_stop:
        print(f'\nEarly Stopping at Epoch {epoch}')
        break

print('=' * 62)
print(f'학습 완료!')
print(f'  최적 Epoch  : {np.argmin(history["val_loss"]) + 1}번째')
print(f'  최적 Val MAE: {min(history["val_loss"]):.5f}')

# 저장했던 최적 모델 가중치 불러오기
model.load_state_dict(torch.load(MODEL_PATH))
print(f'  Best 모델 로드 완료!')


## STEP 7 — 학습 결과 및 예측 시각화

### 왜 시각화가 중요한가?

숫자만 봐서는 모델이 '제대로' 학습됐는지 알기 어렵습니다.  
그래프로 확인해야 할 것들:

**Loss 곡선 보는 법:**
```
정상:           Train loss와 Val loss가 함께 아래로 내려감
과적합:         Train loss는 내려가는데 Val loss가 올라감 (간격이 벌어짐)
과소적합:       둘 다 높은 상태에서 내려가지 않음 (모델이 너무 단순)
```

**예측 vs 실제 비교:**  
빨간선(예측)이 초록선(실제)의 패턴을 잘 따라가는지 확인합니다.


In [ ]:
# ============================================================
# 7-1. Loss 곡선 시각화
# ============================================================

best_epoch = np.argmin(history['val_loss'])  # 검증 손실이 가장 낮은 epoch

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── 학습/검증 Loss 곡선 ─────────────────────────────────
axes[0].plot(history['train_loss'], label='Train MAE', color='steelblue', linewidth=1.5)
axes[0].plot(history['val_loss'],   label='Val MAE',   color='tomato',    linewidth=1.5)
# 최적 epoch 위치에 수직선 표시
axes[0].axvline(best_epoch, color='green', linestyle='--', alpha=0.7,
                label=f'Best Epoch ({best_epoch + 1})')
axes[0].set_title('학습 Loss 곡선 (MAE)', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MAE Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# ── 학습률 변화 ──────────────────────────────────────────
axes[1].plot(history['lr'], color='purple', linewidth=1.5)
axes[1].set_title('Learning Rate 변화 (ReduceLROnPlateau 효과)', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Learning Rate')
axes[1].set_yscale('log')  # 로그 스케일 (작은 변화도 잘 보임)
axes[1].grid(alpha=0.3)
# 계단식으로 줄어드는 게 정상 (ReduceLR 발동 흔적)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'loss_curve.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'Best Epoch : {best_epoch + 1}번째')
print(f'Best Val MAE: {min(history["val_loss"]):.5f}')


In [ ]:
# ============================================================
# 7-2. 샘플 제품 예측 vs 실제 시각화
# ============================================================
# 검증 기간(마지막 30일)에 대해 모델이 얼마나 잘 맞히는지 확인

model.eval()  # 평가 모드 전환 (Dropout 비활성화)

# 판매량이 있는 제품 중 상위 6개 선택
active_products = (
    df[df['qty_log'] > 0].groupby('ID')['qty_log'].sum()
    .nlargest(6).index.tolist()
)

fig, axes = plt.subplots(3, 2, figsize=(16, 12))

with torch.no_grad():  # 예측 시엔 기울기 계산 불필요
    for ax, pid in zip(axes.flatten(), active_products):

        prod_df = df[df['ID'] == pid].sort_values('date').reset_index(drop=True)

        # 검증 기간 시작 위치: 전체 길이 - 검증일수 - 입력길이
        val_start_idx = len(prod_df) - VAL_DAYS - SEQ_LEN
        if val_start_idx < 0:
            ax.set_title(f'ID {pid}: 데이터 부족')
            continue

        # SEQ_LEN일의 피처값을 추출해서 텐서로 변환
        x_input = prod_df[FEATURE_COLS].iloc[val_start_idx : val_start_idx + SEQ_LEN].values
        # unsqueeze(0): (SEQ_LEN, N_FEAT) -> (1, SEQ_LEN, N_FEAT) 배치 차원 추가
        x_tensor = torch.tensor(x_input, dtype=torch.float32).unsqueeze(0).to(DEVICE)

        pred_log = model(x_tensor).cpu().numpy().flatten()  # GPU -> CPU -> numpy
        pred_qty = np.expm1(pred_log)     # log 역변환
        pred_qty = np.maximum(pred_qty, 0) # 음수 판매량 제거

        actual_qty = prod_df['qty'].iloc[
            val_start_idx + SEQ_LEN : val_start_idx + SEQ_LEN + PRED_LEN
        ].values

        # 그래프 그리기
        hist_qty = prod_df['qty'].iloc[val_start_idx : val_start_idx + SEQ_LEN].values
        ax.plot(range(SEQ_LEN), hist_qty,
                color='steelblue', label='입력 (과거 28일)', linewidth=1.2)
        if len(actual_qty) > 0:
            ax.plot(range(SEQ_LEN, SEQ_LEN + len(actual_qty)), actual_qty,
                    color='green', label='실제값', linewidth=1.5, linestyle='--')
        ax.plot(range(SEQ_LEN, SEQ_LEN + PRED_LEN), pred_qty,
                color='tomato', label='예측값', linewidth=1.5)
        ax.axvline(SEQ_LEN - 0.5, color='gray', linestyle=':', alpha=0.7)

        mae_val = mean_absolute_error(
            actual_qty[:len(pred_qty)], pred_qty[:len(actual_qty)]
        ) if len(actual_qty) > 0 else float('nan')

        ax.set_title(f'ID {pid}  |  Val MAE: {mae_val:.2f}', fontsize=9)
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)

plt.suptitle('샘플 제품: 예측 vs 실제 (검증 기간)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'sample_predictions.png'), dpi=150, bbox_inches='tight')
plt.show()


## STEP 8 — 최종 예측 및 제출 파일 생성

### 이 단계에서 하는 일

학습된 모델로 **실제 제출 기간 (2023-04-05 ~ 2023-04-25)** 의  
15,890개 전 제품 판매량을 예측하고 제출용 CSV를 만듭니다.

### 주의 사항
- 예측값이 `log(x+1)` 형태이므로 반드시 **역변환 (`expm1`)** 필요
- 판매량은 음수가 될 수 없으므로 음수는 0으로 처리
- 판매량은 정수이므로 반올림 처리
- `sample_submission.csv`의 제품 순서를 반드시 유지해야 함


In [ ]:
# ============================================================
# 8-1. 전체 제품 예측
# ============================================================

model.eval()
all_preds = {}  # {제품ID : 예측값 배열(21일)} 딕셔너리

product_ids = df['ID'].unique()

with torch.no_grad():
    for pid in tqdm(product_ids, desc='전체 제품 예측 중'):

        prod_df = df[df['ID'] == pid].sort_values('date').reset_index(drop=True)

        # 마지막 SEQ_LEN(28)일의 피처 추출
        if len(prod_df) < SEQ_LEN:
            # 데이터가 28일보다 적은 경우: 앞에 0으로 패딩(채우기)
            pad_len   = SEQ_LEN - len(prod_df)
            feat_vals = np.vstack([
                np.zeros((pad_len, N_FEAT)),
                prod_df[FEATURE_COLS].values
            ])
        else:
            feat_vals = prod_df[FEATURE_COLS].iloc[-SEQ_LEN:].values  # 마지막 28일

        # (SEQ_LEN, N_FEAT) -> (1, SEQ_LEN, N_FEAT) 배치 차원 추가
        x_tensor = torch.tensor(feat_vals, dtype=torch.float32).unsqueeze(0).to(DEVICE)

        pred_log = model(x_tensor).cpu().numpy().flatten()  # (21,)
        pred_qty = np.expm1(pred_log)                       # log 역변환
        pred_qty = np.maximum(np.round(pred_qty), 0).astype(int)  # 음수 제거 & 정수 변환

        all_preds[pid] = pred_qty

print(f'예측 완료: {len(all_preds):,}개 제품')


In [ ]:
# ============================================================
# 8-2. 제출 파일(submission.csv) 생성
# ============================================================

# 딕셔너리 -> DataFrame (행: 제품, 열: 예측 날짜)
pred_df = pd.DataFrame.from_dict(
    all_preds,
    orient='index',      # 딕셔너리의 key(ID)가 행이 됨
    columns=PRED_DATES   # 열 이름을 예측 날짜로 지정
).reset_index()

pred_df.rename(columns={'index': 'ID'}, inplace=True)

# ID 형식 맞추기: 숫자 -> 5자리 문자열 (예: 0 -> '00000')
pred_df['ID'] = pred_df['ID'].astype(str).str.zfill(5)

# sample_submission 순서에 맞춰 행 정렬 (제출 요구사항)
submission_ids = submission['ID'].astype(str).str.zfill(5)
pred_df = pred_df.set_index('ID').reindex(submission_ids).reset_index()

# 결측치 0처리 & 정수형 변환
pred_df = pred_df.fillna(0)
for col in PRED_DATES:
    pred_df[col] = pred_df[col].astype(int)

SUBMIT_PATH = os.path.join(OUTPUT_DIR, 'submission.csv')
pred_df.to_csv(SUBMIT_PATH, index=False)

print(f'제출 파일 저장 완료!')
print(f'  경로 : {SUBMIT_PATH}')
print(f'  Shape: {pred_df.shape}')
pred_df.head()


In [ ]:
# ============================================================
# 8-3. 예측 결과 검증 & 시각화
# ============================================================
# 제출 전 최종 점검: 예측값이 합리적인 범위인지 확인

pred_values = pred_df[PRED_DATES].values.flatten()  # 전체 예측값을 1차원으로

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 예측 판매량 분포
axes[0].hist(pred_values[pred_values > 0], bins=60,
             color='steelblue', alpha=0.8, edgecolor='white')
axes[0].set_title('예측 판매량 분포 (0 제외)', fontsize=13)
axes[0].set_xlabel('예측 판매량')
axes[0].set_ylabel('빈도수')
axes[0].grid(alpha=0.3)

# 예측 기간 일별 총 판매량
daily_pred_sum = pred_df[PRED_DATES].sum(axis=0)
axes[1].bar(range(len(PRED_DATES)), daily_pred_sum.values,
            color='tomato', alpha=0.8, edgecolor='white')
axes[1].set_title(f'예측 기간 일별 총 판매량', fontsize=12)
axes[1].set_xticks(range(len(PRED_DATES)))
axes[1].set_xticklabels([d[5:] for d in PRED_DATES], rotation=45, ha='right', fontsize=8)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'prediction_dist.png'), dpi=150, bbox_inches='tight')
plt.show()

print('=' * 50)
print('📊 예측 결과 최종 요약')
print('=' * 50)
print(f'  총 예측값 합계     : {pred_values.sum():,}개')
print(f'  예측값이 0인 비율  : {(pred_values == 0).mean():.1%}')
print(f'  예측 최댓값        : {pred_values.max():,}개')
print(f'  예측 평균 (0 제외) : {pred_values[pred_values > 0].mean():.1f}개')
print('=' * 50)
print()
print('🎉 모든 과정 완료! submission.csv를 제출하세요.')
print(f'   저장 위치: {SUBMIT_PATH}')
